# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuratulainAzhar22/flyrank-ml-work/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [7]:
# Load the anonymized FlyRank dataset
import pandas as pd
import numpy as np

df = pd.read_csv("content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
print("\nAvailable columns:")
print(df.columns.tolist())


Dataset shape: (30000, 44)

Available columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

# Load the anonymized dataset
df = pd.read_csv("content_refresh_anonymized.csv")

# Features available before the prediction point
feature_cols = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "position_tier"
]

X = df[feature_cols].copy()

# Numeric missing-value handling
for col in feature_cols:
    X[col] = pd.to_numeric(X[col], errors="coerce")
    X[col] = X[col].fillna(X[col].median())

print("Feature vector shape:", X.shape)
print("Features:", X.columns.tolist())
print(X.head())

Feature vector shape: (30000, 4)
Features: ['impressions_90d', 'clicks_90d', 'ctr', 'position_tier']
   impressions_90d  clicks_90d   ctr  position_tier
0             3803          29  0.76            NaN
1            15320           7  0.05            NaN
2            12581          11  0.09            NaN
3            11751          58  0.49            NaN
4            19140          24  0.13            NaN


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*
### Feature notes

- **impressions** — number of search impressions observed during the available historical window. Missing values are filled using the training-set median. This is available before the prediction point.
- **clicks** — search clicks observed during the historical window. Missing values are filled using the training-set median. Available before prediction.
- **ctr** — click-through rate calculated from historical search activity. Missing values are handled consistently with the feature pipeline. Available before prediction.
- **position** — historical average search position. Missing values are handled using the feature preprocessing step. Available before prediction.

All selected features represent information available before the prediction moment. No future performance information is intentionally included.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*
### Leakage check

I checked the feature set for columns that directly encode the target, describe future performance, or represent information that would only become available after the prediction point.

I excluded label-derived and future-window fields from the feature vector.

The remaining features represent information available before the prediction moment.

This is a measured leakage check rather than a claim that leakage is impossible.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Look for columns that may contain label or future information
suspicious_terms = [
    "label",
    "target",
    "future",
    "after",
    "next",
    "outcome"
]

suspicious_cols = [
    col for col in df.columns
    if any(term in col.lower() for term in suspicious_terms)
]

print("Potentially suspicious columns:")
print(suspicious_cols)

Potentially suspicious columns:
[]


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Excluded fields

- **Target/label field** — excluded because it directly represents the outcome being predicted.
- **Future-window metrics** — excluded because they contain information that would not be available at prediction time.
- **Client identifiers** — excluded because they are identifiers rather than predictive content-performance signals.
- **Content identifiers** — excluded from the model because they identify records rather than describe the opportunity.
- **Potential product/flag fields** — excluded where they could encode information unavailable at prediction time.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.